In [ ]:
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.metrics import r2_score
import numpy as np
import random
from torch.cuda.amp import GradScaler, autocast
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Random seed configuration
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Model definitions
class SimpleGNN(nn.Module):
    """GCN teacher trained on a chemistry-stratified subset."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)
        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if hasattr(self, 'edge_norm') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        if hasattr(self, 'global_norm') and u is not None:
            u = self.global_norm(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        node_pool = global_mean_pool(x, data.batch)
        h = torch.cat([node_pool, self.global_mlp(u)], dim=1) if u is not None else node_pool
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out

class EnhancedGNN(nn.Module):
    """Student GCN used for knowledge distillation and inference."""
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None
        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
        self.convs = nn.ModuleList()
        in_dim = node_dim
        for h in hidden_dims:
            self.convs.append(GCNConv(in_dim, h))
            in_dim = h
        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim//2, 1)
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)
        if self.edge_norm and hasattr(data, 'edge_attr') and data.edge_attr is not None:
            _ = self.edge_norm(data.edge_attr)
        u = getattr(data, 'u', None)
        gf = None
        if u is not None and self.global_norm is not None:
            u = self.global_norm(u)
            gf = self.global_mlp(u)
        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)
        pooled = global_mean_pool(x, data.batch)
        h = torch.cat([pooled, gf], dim=1) if gf is not None else pooled
        out = self.output_mlp(h).squeeze()
        return (out, h) if return_feat else out

class GateNet(nn.Module):
    """Compute sample-dependent weights over the retained teachers."""
    def __init__(self, in_dim, hidden_dim, num_teachers):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_teachers)

    def forward(self, h):
        a = F.relu(self.fc1(h))
        return F.softmax(self.fc2(a), dim=-1)

class Adapter(nn.Module):
    """Map the student feature representation to the teacher feature space."""
    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)

    def forward(self, h):
        return self.linear(h)

# Convert precomputed graph dictionaries into PyG mini-batches.
def create_data_loader(graph_list, batch_size=32, shuffle=True):
    data_list = []
    for g in graph_list:
        data_list.append(Data(
            x=g['x'],
            edge_index=g['edge_index'],
            edge_attr=g.get('edge_attr', None),
            u=g.get('u', None),
            y=g['y'],
            y_soft=g.get('y_soft', None)
        ))
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

# Distillation training and checkpoint selection
def train_model(
    train_dir, val_dir, teacher_paths, save_path,
    gate_hidden=128, hint_lambda=5.0, weight_ratio=(0.6,0.4),
    hidden_dims=[128,128], dropout=0.1,
    epochs=500, batch_size=64, lr=1e-3, min_lr=1e-4,
    lr_patience=20, es_patience=50,
    seed=42, exclude_teacher_idx=None
):
    """
    exclude_teacher_idx: Index of the omitted teacher; its soft-label column is removed to match the retained teacher models.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_seed(seed)

    # Load precomputed molecular graph data
    train_g = torch.load(os.path.join(train_dir, 'graph_data.pt'))
    val_g = torch.load(os.path.join(val_dir, 'graph_data.pt'))

    g0 = train_g[0]
    print(f"Node feature dimension: {g0['x'].shape[1]}, edge features available: {('edge_attr' in g0 and g0['edge_attr'] is not None)}, "
          f"global features available: {('u' in g0 and g0['u'] is not None)}")

    # Verify the shape of the five-column teacher soft labels
    shapes_train = {}
    for g in train_g:
        s = g['y_soft'].shape
        shapes_train[s] = shapes_train.get(s, 0) + 1
    shapes_val = {}
    for g in val_g:
        s = g['y_soft'].shape
        shapes_val[s] = shapes_val.get(s, 0) + 1
    print(f"Training soft-label shape counts: {shapes_train}")
    print(f"Validation soft-label shape counts: {shapes_val}")
    assert len(shapes_train) == 1, "Inconsistent soft-label shapes in the training data."
    assert list(shapes_train.keys())[0][-1] == 5, "Training soft labels must contain exactly five teacher columns."
    assert len(shapes_val) == 1, "Inconsistent soft-label shapes in the validation data."
    assert list(shapes_val.keys())[0][-1] == 5, "Validation soft labels must contain exactly five teacher columns."

    # Standardize hard targets using training-set statistics only
    ys = torch.stack([g['y'] for g in train_g]).view(-1)
    y_mean, y_std = ys.mean().item(), ys.std().item() + 1e-8
    for g in train_g:
        g['y'] = (g['y'] - y_mean) / y_std
        g.setdefault('y_soft', g['y'])
    for g in val_g:
        g['y'] = (g['y'] - y_mean) / y_std
        g.setdefault('y_soft', g['y'])

    # Remove the excluded teacher column from the soft-label matrix
    if exclude_teacher_idx is not None:
        def trim_soft_label(g):
            soft = g['y_soft']
            if soft.dim() == 0 or (soft.dim() == 1 and soft.numel() == 1):
                return   # Leave scalar placeholders unchanged.
            K_all = soft.size(-1)
            cols = [i for i in range(K_all) if i != exclude_teacher_idx]
            g['y_soft'] = soft[..., cols]
        for g in train_g:
            trim_soft_label(g)
        for g in val_g:
            trim_soft_label(g)
        sample_shape = train_g[0]['y_soft'].shape
        print(f"Removed soft-label column {exclude_teacher_idx}; new shape: {sample_shape} (remaining teachers: {sample_shape[-1]}).")

    tr_loader = create_data_loader(train_g, batch_size, True)
    va_loader = create_data_loader(val_g, batch_size, False)

    # Student GNN (GCN backbone)
    sample = train_g[0]
    n_dim = sample['x'].size(1)
    e_dim = sample['edge_attr'].size(1) if sample.get('edge_attr') is not None else 0
    g_dim = sample['u'].size(1) if sample.get('u') is not None else 0
    student = EnhancedGNN(n_dim, e_dim, g_dim, hidden_dims, dropout).to(device)

    # Load the retained pretrained teacher checkpoints
    teachers = []
    for p in teacher_paths:
        ck = torch.load(p, map_location=device)
        t = SimpleGNN(ck['node_dim'], ck.get('edge_dim',0), ck.get('global_dim',0),
                      ck['hidden_dims'], ck['dropout']).to(device)
        t.load_state_dict(ck['model_state_dict'], strict=False)
        t.eval()
        for param in t.parameters():
            param.requires_grad = False
        teachers.append(t)
    print(f"Loaded {len(teachers)} pretrained teacher models.")

    K = len(teachers)
    gate = GateNet(student.final_dim, gate_hidden, K).to(device)
    adapter = Adapter(student.final_dim, student.final_dim).to(device)

    optimizer = optim.Adam(
        list(student.parameters()) + list(gate.parameters()) + list(adapter.parameters()),
        lr=lr, weight_decay=1e-5
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 'min', factor=0.5, patience=lr_patience, min_lr=min_lr
    )
    scaler = GradScaler(enabled=(device.type == 'cuda'))

    best_r2 = -1e9
    patience = 0
    history = {'loss': [], 'train_r2': [], 'val_r2': []}

    def eval_loader(loader):
        student.eval()
        ys, ps = [], []
        with torch.no_grad():
            for b in loader:
                b = b.to(device)
                out, _ = student(b, return_feat=True)
                ys.append(b.y.view(-1).cpu().numpy())
                ps.append(out.cpu().numpy())
        return r2_score(np.concatenate(ys), np.concatenate(ps))

    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = 0.0
        for batch in tr_loader:
            batch = batch.to(device)
            with autocast(enabled=(device.type == 'cuda')):
                pred_s, h_s = student(batch, return_feat=True)
                Ht = torch.stack([t(batch, return_feat=True)[1] for t in teachers], dim=1)
                w = gate(h_s)
                Ht_g = (w.unsqueeze(-1) * Ht).sum(dim=1)
                loss_hint = F.mse_loss(adapter(h_s), Ht_g)

                fused = (w * batch.y_soft).sum(dim=1)      # The gate weights and soft labels have matching teacher dimensions.
                pred_f = weight_ratio[0] * pred_s + weight_ratio[1] * fused
                loss = (weight_ratio[0] * F.mse_loss(pred_f, batch.y.view(-1)) +
                        weight_ratio[1] * F.mse_loss(pred_s, fused) +
                        hint_lambda * loss_hint)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        train_r2 = eval_loader(tr_loader)
        val_r2 = eval_loader(va_loader)
        avg_loss = total_loss / len(tr_loader)

        history['loss'].append(avg_loss)
        history['train_r2'].append(train_r2)
        history['val_r2'].append(val_r2)

        if epoch % 30 == 0:
            print(f"Epoch {epoch:4d} | Loss {avg_loss:.4f} | Train R² {train_r2:.4f} | Val R² {val_r2:.4f} | LR {optimizer.param_groups[0]['lr']:.2e}")

        scheduler.step(avg_loss)

        if val_r2 > best_r2:
            best_r2 = val_r2
            patience = 0
            torch.save({
                'model_state_dict': student.state_dict(),
                'gate_state': gate.state_dict(),
                'adapter_state': adapter.state_dict(),
                'y_mean': y_mean,
                'y_std': y_std,
                'history': history,
                'node_dim': n_dim,
                'edge_dim': e_dim,
                'global_dim': g_dim,
                'hidden_dims': hidden_dims,
                'dropout': dropout
            }, save_path)
            if epoch % 30 != 0:
                print(f"Saved best model at Epoch {epoch}")
        else:
            patience += 1
            if patience >= es_patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    print(f"Training complete; best validation R² = {best_r2:.4f}")

    return save_path

# Main experiment: leave-one-teacher-out ablation
if __name__ == '__main__':
    seeds = [8, 618, 1189, 2077, 2333]

    # Use the fixed, seed-specific hyperparameters selected in the original experiment.
    # See config/kminus1_seed_hyperparameters.csv for the exact parameter values.
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    config_path = os.path.join(PROJECT_ROOT, "config", "kminus1_seed_hyperparameters.csv")
    seed_distillation_config = {}
    with open(config_path, newline='', encoding='utf-8') as config_file:
        for row in csv.DictReader(config_file):
            seed = int(row['seed'])
            if seed in seed_distillation_config:
                raise ValueError(f"Duplicate hyperparameter configuration for seed {seed}.")
            seed_distillation_config[seed] = {
                'hint_lambda': float(row['hint_lambda']),
                'weight_ratio': (float(row['weight_student']), float(row['weight_teacher'])),
                'gate_hidden': int(row['gate_hidden']),
                'config_id': row['config_id'],
            }
    if set(seed_distillation_config) != set(seeds):
        raise ValueError("The seed-specific hyperparameter file must match the seed list.")

    train_dir = os.path.join(PROJECT_ROOT, "data-set", "train")
    val_dir = os.path.join(PROJECT_ROOT, "data-set", "validation")

    # The checkpoint order must match the columns of y_soft.
    teacher_checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints", "teachers")
    teacher_order = ("qcut", "elem", "molwt", "fp", "scaffold")
    all_teacher_paths = [
        os.path.join(teacher_checkpoint_dir, f"{teacher_name}.pt")
        for teacher_name in teacher_order
    ]

    save_root = os.path.join(PROJECT_ROOT, "results", "kminus1_ablation")
    os.makedirs(save_root, exist_ok=True)

    epochs = 1000
    batch_size = 64
    lr = 1e-3
    min_lr = 5e-5
    lr_patience = 30
    es_patience = 100
    hidden_dims = [128, 128]
    dropout = 0.1

    # Record the generated checkpoints without a separate predictive-metrics report.
    manifest_path = os.path.join(save_root, 'checkpoint_manifest.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as manifest_file:
        manifest_writer = csv.DictWriter(
            manifest_file,
            fieldnames=['excluded_teacher', 'seed', 'config_id', 'checkpoint_path']
        )
        manifest_writer.writeheader()

        for exclude_idx, excluded_teacher in enumerate(teacher_order):
            teacher_paths = [
                path for index, path in enumerate(all_teacher_paths)
                if index != exclude_idx
            ]
            ablation_save_dir = os.path.join(save_root, f"exclude_{excluded_teacher}")
            os.makedirs(ablation_save_dir, exist_ok=True)
            print(f"\nExcluded teacher: {excluded_teacher}; retained teachers: {len(teacher_paths)}")

            for seed in seeds:
                seed_cfg = seed_distillation_config[seed]
                combo_key = seed_cfg['config_id']
                save_path = os.path.join(
                    ablation_save_dir, f"student_{combo_key}_seed{seed}.pt"
                )
                print(f"Training seed {seed} with fixed configuration {combo_key}.")
                set_seed(seed)
                train_model(
                    train_dir=train_dir,
                    val_dir=val_dir,
                    teacher_paths=teacher_paths,
                    save_path=save_path,
                    gate_hidden=seed_cfg['gate_hidden'],
                    hint_lambda=seed_cfg['hint_lambda'],
                    weight_ratio=seed_cfg['weight_ratio'],
                    hidden_dims=hidden_dims,
                    dropout=dropout,
                    epochs=epochs,
                    batch_size=batch_size,
                    lr=lr,
                    min_lr=min_lr,
                    lr_patience=lr_patience,
                    es_patience=es_patience,
                    seed=seed,
                    exclude_teacher_idx=exclude_idx,
                )
                manifest_writer.writerow({
                    'excluded_teacher': excluded_teacher,
                    'seed': seed,
                    'config_id': combo_key,
                    'checkpoint_path': os.path.relpath(save_path, PROJECT_ROOT),
                })
                manifest_file.flush()

    print(f"\nCompleted all leave-one-teacher-out runs. Manifest: {manifest_path}")